# Dockerfile and Custom Images


## What is a Dockerfile?

A **Dockerfile** is a text file that describes how to build a Docker image.

It answers questions like:

- Which base image should we start from?
- Which files should be copied into the image?
- Which packages should be installed?
- What command should run when the container starts?

Simple flow:

```text
Dockerfile + project files --docker build--> image --docker run--> container
```


## Build Time vs Run Time

Two phases are important:

| Phase | Command | What happens |
|-------|---------|--------------|
| Build time | `docker build` | Docker creates an image from the Dockerfile. |
| Run time | `docker run` | Docker starts a container from an image. |

Example:

```bash
docker build -t my_python_app .
docker run my_python_app
```

A common beginner mistake is confusing build-time commands with run-time commands.


## Important Dockerfile Commands

| Command | Purpose |
|---------|---------|
| `FROM` | Choose a base image. |
| `WORKDIR` | Set working directory inside the image/container. |
| `COPY` | Copy files from host into image. |
| `RUN` | Execute commands while building the image. |
| `EXPOSE` | Document which port the app uses. |
| `CMD` | Default command when a container starts. |
| `ENTRYPOINT` | Fixed executable for the container. |

`RUN` happens during image build. `CMD` happens when a container starts.


## Example Python Application

Create a simple `app.py`:

```python
from http.server import BaseHTTPRequestHandler, HTTPServer

class Handler(BaseHTTPRequestHandler):
    def do_GET(self):
        self.send_response(200)
        self.end_headers()
        self.wfile.write(b"Hello from Docker")

HTTPServer(("0.0.0.0", 5000), Handler).serve_forever()
```

The app listens on `0.0.0.0`, not `127.0.0.1`, so it can receive traffic from outside the container.


## Basic Dockerfile for Python

```dockerfile
FROM python:3.12-slim

WORKDIR /app

COPY . /app

EXPOSE 5000

CMD ["python", "app.py"]
```

Explanation:

| Line | Meaning |
|------|---------|
| `FROM python:3.12-slim` | Start from a Python image. |
| `WORKDIR /app` | Use `/app` as the working directory. |
| `COPY . /app` | Copy current project files into the image. |
| `EXPOSE 5000` | Document that the app listens on port `5000`. |
| `CMD [...]` | Run the Python app when the container starts. |

`EXPOSE` documents the container port, but it does not publish the port to your host. You still need `docker run -p`.


## Example Dockerfile for Django/DRF

A Django or DRF project usually runs with `manage.py`.

Example project structure:

```text
my_project/
    manage.py
    requirements.txt
    config/
        settings.py
        urls.py
        wsgi.py
```

Development Dockerfile:

```dockerfile
FROM python:3.12-slim

ENV PYTHONDONTWRITEBYTECODE=1
ENV PYTHONUNBUFFERED=1

WORKDIR /app

COPY requirements.txt /app/
RUN pip install --no-cache-dir -r requirements.txt

COPY . /app/

EXPOSE 8000

CMD ["python", "manage.py", "runserver", "0.0.0.0:8000"]
```

The two Python environment options are common in Docker examples:

| Option | Meaning | Why useful in Docker? |
|--------|---------|-----------------------|
| `PYTHONDONTWRITEBYTECODE=1` | Python does not create `.pyc` / `__pycache__` files. | Keeps the container filesystem cleaner. |
| `PYTHONUNBUFFERED=1` | Python prints output immediately instead of buffering it. | Logs appear immediately in `docker logs`. |

They are not required for a beginner example, but they are useful and common enough to keep.

Build the image:

```bash
docker build -t my_django_app .
```

Run the container:

```bash
docker run -p 8000:8000 my_django_app
```

Open:

```text
http://localhost:8000
```

Important notes:

- Use `0.0.0.0:8000`, not `127.0.0.1:8000`, so the app can receive traffic from outside the container.
- This example is for development/learning.
- For production, use Gunicorn instead of Django's `runserver`.

Production-style command example:

```dockerfile
CMD ["gunicorn", "config.wsgi:application", "--bind", "0.0.0.0:8000"]
```

Here `config` should be replaced with the package that contains your project's `wsgi.py` file.


## Build and Run

Build the image:

```bash
docker build -t my_python_app .
```

Run the container:

```bash
docker run -d -p 5000:5000 my_python_app
```

Options:

| Option | Meaning |
|--------|---------|
| `-d` | detached mode; run in background |
| `-p 5000:5000` | map host port 5000 to container port 5000 |
| `my_python_app` | image name |

Open:

```text
http://localhost:5000
```


## Port Mapping

A container has its own network namespace. If an app listens inside the container, your host machine cannot automatically access it.

Use `-p`:

```bash
docker run -p HOST_PORT:CONTAINER_PORT image_name
```

Example:

```bash
docker run -p 8000:5000 my_python_app
```

This means:

```text
localhost:8000 on host → port 5000 inside container
```

`EXPOSE` in Dockerfile documents the intended port, but `-p` actually publishes it.


## Installing Dependencies

For Python projects, dependencies usually live in `requirements.txt`:

```text
Django==5.0.0
djangorestframework==3.15.0
```

Dockerfile:

```dockerfile
FROM python:3.12-slim

WORKDIR /app

COPY requirements.txt /app/
RUN pip install --no-cache-dir -r requirements.txt

COPY . /app

CMD ["python", "app.py"]
```

Why copy `requirements.txt` first?

Docker caches image layers. If requirements do not change, Docker can reuse the dependency-install layer.


## Small Curious Note: Build Cache

Docker tries to reuse previous build steps when files have not changed.

That is why this order is useful:

```dockerfile
COPY requirements.txt /app/
RUN pip install --no-cache-dir -r requirements.txt
COPY . /app/
```

If only your Python code changes, Docker may reuse the dependency installation step.

You do not need to deeply understand image layers now. Just remember:

> Put dependency installation before copying all source code when possible.


## `.dockerignore`

`.dockerignore` tells Docker which files should not be copied into the build context.

Example:

```dockerignore
__pycache__/
*.pyc
*.pyo
.git/
.env
.env.*
venv/
.env/
.pytest_cache/
.mypy_cache/
```

Why it matters:

- Smaller build context.
- Faster builds.
- Avoid leaking secrets.
- Avoid copying local virtual environments.
- Avoid unnecessary cache invalidation.


## CMD vs ENTRYPOINT

`CMD` gives the default command:

```dockerfile
CMD ["python", "app.py"]
```

You can override it:

```bash
docker run my_python_app python --version
```

`ENTRYPOINT` is more fixed:

```dockerfile
ENTRYPOINT ["python"]
CMD ["app.py"]
```

For beginner app Dockerfiles, `CMD` is usually enough.


## Inspecting and Debugging Images

List images:

```bash
docker images
```

Run a shell inside an image:

```bash
docker run -it my_python_app bash
```

If the image does not have Bash:

```bash
docker run -it my_python_app sh
```

View logs of a running container:

```bash
docker logs CONTAINER_ID
```


## Optional: Tag and Publish Later

For this course, building and running locally is enough:

```bash
docker build -t my_python_app .
docker run -p 5000:5000 my_python_app
```

Later, in real teams, images can be pushed to Docker Hub or another registry:

```bash
docker tag my_python_app username/my_python_app:latest
docker push username/my_python_app:latest
```

This is useful for CI/CD and deployment, but it is optional for now.


## Common Beginner Mistakes

| Mistake | Result | Fix |
|---------|--------|-----|
| App listens on `127.0.0.1` inside container | Port mapping does not work from host | Listen on `0.0.0.0` |
| Forgetting `-p` | App runs but cannot be reached | Use `-p HOST:CONTAINER` |
| Copying `.env` into image | Secrets may leak | Add `.env` to `.dockerignore` |
| Installing deps after copying everything | Cache is less useful | Copy `requirements.txt` first |
| Expecting container filesystem to persist | Data disappears after removal | Use volumes |
| Confusing image and container | Wrong command usage | Remember: image is template, container is instance |


## Summary

- A Dockerfile defines how to build an image.
- `RUN` executes during build; `CMD` executes when the container starts.
- Use `docker build` to create an image.
- Use `docker run` to create/start a container.
- Use `-p` to publish ports.
- Use `.dockerignore` to keep builds small and secrets safe.
- Docker cache works best when dependency files are copied before the full source code.
